# 06 — Polymer chains: build molecules with numpy

The bead-spring (Kremer–Grest) polymer is *the* model of polymer physics:
beads connected by FENE bonds, purely repulsive LJ between everything, a
Langevin thermostat as implicit solvent. We build the chains ourselves —
a LAMMPS data file is just text, and thanks to the shared filesystem
LAMMPS reads what Python writes with plain `open()`.

The physics payoff: a self-avoiding chain of $N$ beads swells as
$R_g \sim N^{\nu}$ with the Flory exponent $\nu \approx 0.588$ —
measurably bigger than the random-walk value $1/2$.

In [ ]:
%pip install lammps-js matplotlib

## A data file from numpy

`make_chain(N)` writes an `N`-bead zig-zag chain (bond lengths near the
FENE minimum) with `N−1` bonds. Note the file is created with ordinary
Python `open()` — check the file browser after running:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps

def make_chain(N, path):
    lines = [f"bead-spring chain, N = {N}", "",
             f"{N} atoms", f"{N - 1} bonds",
             "1 atom types", "1 bond types", "",
             "-60 60 xlo xhi", "-60 60 ylo yhi", "-60 60 zlo zhi", "",
             "Masses", "", "1 1.0", "", "Atoms", ""]
    for i in range(N):
        lines.append(f"{i + 1} 1 1 {i * 0.93:.3f} {0.15 * (i % 2):.3f} 0.0")
    lines += ["", "Bonds", ""]
    for i in range(N - 1):
        lines.append(f"{i + 1} 1 {i + 1} {i + 2}")
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")

make_chain(16, "chain.data")
print(open("chain.data").read()[:180], "…")

## Simulate one chain length

Standard Kremer–Grest parameters: FENE with $k = 30$, $R_0 = 1.5$, and a
WCA (purely repulsive, cut at $2^{1/6}$) pair interaction. `compute
gyration` reports $R_g$; we sample it every 500 steps after an
equilibration proportional to the chain's relaxation time. The longer
chains take a minute — progress prints as it goes:

In [ ]:
async def gyration(N):
    make_chain(N, "chain.data")
    lmp = await lammps(output=None)
    lmp.commands_string(f"""
units         lj
atom_style    bond
special_bonds fene
read_data     chain.data
bond_style    fene
bond_coeff    1 30.0 1.5 1.0 1.0
pair_style    lj/cut 1.122462
pair_coeff    1 1 1.0 1.0
pair_modify   shift yes
neigh_modify  every 1 delay 0
timestep      0.005
fix           1 all nve
fix           lang all langevin 1.0 1.0 2.0 {90429 + N}
compute       g all gyration
variable      rg equal c_g
run           {1000 * N}
""")
    samples = []
    for _ in range(4 * N):
        lmp.command("run 500")
        samples.append(lmp.extract_variable("rg"))
    lmp.close()
    return float(np.mean(samples)), float(np.std(samples))

Ns = [8, 16, 32, 48]
rg = {}
for N in Ns:
    rg[N] = await gyration(N)
    print(f"N = {N:>3}:  Rg = {rg[N][0]:.3f} ± {rg[N][1]:.3f}")

## The Flory exponent

Fit $\log R_g$ against $\log(N-1)$ (the number of bonds). Self-avoiding
walks in 3D give $\nu \approx 0.588$; an ideal random walk would give
$1/2$. Short chains overshoot the asymptotic exponent a little — real
(and simulated) polymer physics has finite-size corrections:

In [ ]:
n_bonds = np.array([N - 1 for N in Ns], dtype=float)
rg_mean = np.array([rg[N][0] for N in Ns])
nu, log_pref = np.polyfit(np.log(n_bonds), np.log(rg_mean), 1)

n_line = np.linspace(n_bonds[0], n_bonds[-1], 50)
plt.figure(figsize=(6, 4))
plt.errorbar(n_bonds, rg_mean, yerr=[rg[N][1] for N in Ns], fmt="o",
             capsize=3, label="measured")
plt.plot(n_line, np.exp(log_pref) * n_line**nu, "-",
         label=f"fit: ν = {nu:.3f}")
plt.plot(n_line, rg_mean[0] * (n_line / n_bonds[0])**0.588, "--",
         label="SAW theory: ν = 0.588")
plt.plot(n_line, rg_mean[0] * (n_line / n_bonds[0])**0.5, ":",
         label="ideal chain: ν = 0.5")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("number of bonds N − 1"); plt.ylabel("radius of gyration $R_g$")
plt.legend(fontsize=9); plt.tight_layout(); plt.show()

The excluded-volume repulsion visibly beats the random walk — the chain
*swells*. From here it's a short path to real soft-matter simulations:
many chains in a melt (where, beautifully, ν drops back to ½), grafted
brushes, or pulling a single chain by its ends with `fix spring`.

That's the end of this series — back to [the index](../index.ipynb).